# Chapter 6 — Symmetrisation Operators

Companion notebook to Chapter 6.

Reproduces:
- Figure 6.1: All four symmetrisation operators applied to the same asymmetric k-NN affinity.
- Figure 6.2: Sinkhorn convergence dynamics.
- Figure 6.3: Spectral clustering on each symmetrised graph.

Verifies that all four operators produce symmetric output (Section 6.5).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabkernels.symmetrizers import (
    AdditiveSymmetrizer, MaxOrSymmetrizer, MutualAndSymmetrizer, SinkhornSymmetrizer,
)
from tabkernels.sparsifiers import KNNSparsifier

torch.manual_seed(42); np.random.seed(42)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Figure 6.1: Four symmetrisation operators
Build an asymmetric affinity from hard k-NN sparsification of an RBF Gram matrix; apply each operator.

In [ ]:
N = 30
X = torch.randn(N, 2)
sq = ((X[:, None] - X[None, :]) ** 2).sum(-1)
W_full = torch.exp(-sq / 0.6)
W_asym = KNNSparsifier(k=4, exclude_self=True)(W_full)

ops = {
    'Original (asymmetric)': W_asym,
    'Additive':              AdditiveSymmetrizer()(W_asym),
    'OR (max)':              MaxOrSymmetrizer()(W_asym),
    'AND (mutual)':          MutualAndSymmetrizer()(W_asym),
    'Sinkhorn':              SinkhornSymmetrizer(iters=20)(W_asym + 1e-6),
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, M) in zip(axes, ops.items()):
    sym_diff = (M - M.T).abs().max().item()
    nz = (M > 1e-6).float().mean().item()
    ax.imshow(M.detach(), cmap='viridis', aspect='auto')
    ax.set_title(f'{name}\nsym? {sym_diff < 1e-5} | nz {nz:.2f}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Figure 6.1: Symmetrisation operators applied to a k-NN affinity')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_06_01_symmetrisers.pdf', bbox_inches='tight')
plt.show()

## Figure 6.2: Sinkhorn convergence dynamics

In [ ]:
torch.manual_seed(0)
W = torch.randn(20, 20).abs() + 0.1
iters_to_track = [1, 2, 5, 10, 30]
row_sums_history = []
col_sums_history = []
for n in iters_to_track:
    s = SinkhornSymmetrizer(iters=n)
    out = s(W)
    row_sums_history.append(out.sum(dim=-1).std().item())
    col_sums_history.append(out.sum(dim=0).std().item())
fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.plot(iters_to_track, row_sums_history, 'o-', label='row-sum std')
ax.plot(iters_to_track, col_sums_history, 's-', label='col-sum std')
ax.set_xlabel('Sinkhorn iterations'); ax.set_ylabel('std of marginals')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_title('Figure 6.2: Sinkhorn iteration dynamics — marginal std decays with iters')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_06_02_sinkhorn_convergence.pdf', bbox_inches='tight')
plt.show()

## Figure 6.3: Spectral clustering on each symmetrised graph

In [ ]:
from sklearn.cluster import SpectralClustering

# Build a 2-cluster point set.
torch.manual_seed(0)
Xa = torch.randn(20, 2) + torch.tensor([2.0, 0.0])
Xb = torch.randn(20, 2) + torch.tensor([-2.0, 0.0])
X = torch.cat([Xa, Xb])
y_true = np.concatenate([np.zeros(20), np.ones(20)])

sq = ((X[:, None] - X[None, :]) ** 2).sum(-1)
W_full = torch.exp(-sq / 1.0)
W_knn = KNNSparsifier(k=5, exclude_self=True)(W_full)

ops_for_clustering = {
    'Additive':       AdditiveSymmetrizer()(W_knn),
    'OR':             MaxOrSymmetrizer()(W_knn),
    'AND':            MutualAndSymmetrizer()(W_knn),
    'Sinkhorn':       SinkhornSymmetrizer(iters=20)(W_knn + 1e-6),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (name, W) in zip(axes, ops_for_clustering.items()):
    W_np = W.detach().numpy()
    np.fill_diagonal(W_np, 0)  # spectral clustering convention
    try:
        sc = SpectralClustering(n_clusters=2, affinity='precomputed', random_state=42)
        labels = sc.fit_predict(W_np)
        # Match labels to truth.
        if (labels != y_true).mean() > 0.5:
            labels = 1 - labels
        acc = (labels == y_true).mean()
    except Exception as e:
        labels = np.zeros(len(X)); acc = 0.0
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='RdBu', s=30, edgecolors='k', linewidth=0.4)
    ax.set_title(f'{name} (acc {acc:.2f})')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Figure 6.3: Spectral clustering accuracy on each symmetrised graph')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_06_03_clustering.pdf', bbox_inches='tight')
plt.show()

**End of notebook.**